# Array 개념: `cutlass.Array`와 GPU 메모리 공간

`cutlass.Array`는 이 과정의 핵심 메모리 타입입니다: GPU 메모리를 가리키는 타입 있는 핸들이고, 자기
**shape**, **stride**, **dtype**, **메모리 공간**을 압니다. 핸들 하나로 스칼라 `a[i]`, 다차원
`a[r, c]`, 오프셋 `subview` 접근을 전부 처리합니다 — 손으로 쓰는 포인터 산술이 없습니다.

나머지 절반은 메모리가 *어디에* 있느냐입니다. `Array` 타입 하나가 GPU 메모리 공간 네 가지를 덮고,
그 공간이 **누가 소유하는지, 얼마나 오래 사는지, 어느 스레드가 보는지**를 정합니다. 이 챕터는 커널
두 개로 진행합니다: 하나는 커널 **안에서** 배열을 할당하고, 다른 하나는 모듈 스코프의 커널 **밖에**
선언한 배열을 씁니다. 각각 자기가 만지는 배열을 출력합니다.

원문: [`NVIDIA/cutlass` examples/python/CuTeDSL/notebooks/2_primitives/01_array_concepts.ipynb](https://github.com/NVIDIA/cutlass/blob/098de2a652cf8f00fd70b2df54051c7eccbb855a/examples/python/CuTeDSL/notebooks/2_primitives/01_array_concepts.ipynb) (BSD-3-Clause, commit `098de2a` 기준)

**배울 내용:** DSL의 중심 데이터 타입 **`cutlass.Array`**. 커널 인자는 Array입니다; **슬라이스 =
벡터화 load/store** 관용구(`a[idx:V]`는 정렬을 약속하면 연속된 `V`개 원소를 넓은 트랜잭션 한 번에
옮깁니다); Array의 출력된 레이아웃(dtype, 공간, shape, 정렬) 읽는 법; **local / shared / global /
constant** 메모리 공간과 각각의 소유권·수명·가시성; 커널 안에서 **local**·**shared** 스크래치를
할당하는 법(shared를 안전하게 만드는 배리어 포함); 모듈 스코프에서 **global**·**constant** 정적
변수를 선언하는 법; 그리고 `subview`. 그 과정에서 벡터화된 벡터 덧셈을 완성합니다.

**실행 환경:** 이 노트북은 커널을 *실제로 launch합니다*. 노트북4와 마찬가지로 **`CUTE_DSL_ARCH`
오버라이드를 넣지 않습니다**(오버라이드하면 L4에서 못 도는 cubin이 만들어집니다). `Arch` enum이
`sm_80`부터 시작하므로 T4는 해당 없습니다(README 참고). 이 노트북은 **Colab L4**에서 실행해
검증했습니다. (레지스터 *값*으로서의 Vector는 `02_vector_concepts`, `cute.Tensor` → `cutlass.Array`
다리는 `03_cute_interop`에서 다룹니다 — 원문 `2_primitives` 경로 기준 이름입니다.)

## 0. 환경 확인

재현성을 위해 먼저 GPU 정보를 남깁니다.

In [ ]:
!nvidia-smi

## 0-1. CuTe DSL 설치

`nvidia-cutlass-dsl==4.8.0`으로 고정합니다(노트북1과 동일 버전).

**함정(노트북1에서 이미 확인함): 같은 커널에서 설치 직후 import하면 실패합니다.** `.pth`
파일이 인터프리터 시작 시점에만 읽히기 때문 — 설치 셀 다음에 반드시 `site.main()` 셀을
넣습니다.

In [ ]:
!pip install -q nvidia-cutlass-dsl==4.8.0

In [ ]:
import site
site.main()  # 방금 설치된 nvidia-cutlass-dsl의 .pth 경로를 sys.path에 반영

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## 네 가지 메모리 공간

`cutlass.Array(dtype, size, space=...)`는 **공간**에 따라 할당자를 고릅니다. 각각 익숙한 CUDA C++
선언에 대응합니다:

| `cutlass.Array(...)` | CUDA C++ 대응 | 공간 | 누가 보나 / 수명 |
|---|---|---|---|
| `cutlass.Array(f32, N)` | `float a[N];` | **local** (기본값) | **스레드**마다 비공개 사본 하나 |
| `cutlass.Array(f32, N, space=…smem)` | `__shared__ float a[N];` | **shared** | **블록**마다 사본 하나 |
| `cutlass.Array(f32, N, space=…gmem, name="g")` | `__device__ float g[N];` | **global** | **정적** 버퍼 하나; 모든 스레드 + 모든 launch가 공유 |
| `cutlass.Array(f32, N, space=…cmem, name="c", init=…)` | `__constant__ float c[N];` | **constant** | 읽기 전용 정적, 모든 스레드에 브로드캐스트 |

**local**과 **shared**는 커널 **안에서** 할당하는 스크래치이고, **global**과 **constant**는 모듈
스코프에서 선언해 이름으로 쓰는 **정적 변수**입니다. `size`는 항상 컴파일 타임 상수입니다 — 이들은
정적 버퍼이지, 스레드별 `malloc`이 아닙니다. Global 메모리는 흔한 방식으로도 들어옵니다: **커널
인자**(`from_dlpack`으로 넘긴 텐서)로.

## 1. 스레드 하나, 원소 벡터 하나

커널이 세 줄이라서 설계 결정은 하나뿐입니다: *스레드 하나가 일을 얼마나 하느냐*. 스레드마다 **연속된**
원소 `V`개짜리 윈도우를 주고, 윈도우 전체를 한 번에 처리하게 합니다.

메커니즘은 슬라이스입니다. `a[idx:V]`는 `idx`부터 `V`개를 읽고, `c[idx:V]`에 대입하면 그대로 씁니다.
두 번째 숫자는 파이썬의 stop 인덱스가 아니라 **개수**입니다 -- `a[idx:V]`는 "`idx`부터 `V`개"이지
"`idx`부터 `V` 직전까지"가 아닙니다. 그래서 `[idx, idx+V)` 구간을 맡은 스레드는 입력 둘을 로드하고,
레지스터끼리 더하고, 결과를 저장합니다.

그 슬라이스가 **넓은 트랜잭션 한 번**이 되느냐 스칼라 `V`번이 되느냐는 **정렬**에 달렸습니다. `f32`
네 개를 한 번에 읽는 128비트 로드는 16바이트 정렬된 주소에서만 합법이고, 컴파일러는 그걸 저절로
가정하지 않습니다. 약속은 버퍼를 감쌀 때 합니다(아래의 `from_dlpack(a, assumed_align=16)`). 그리고
실제로 성립합니다: 모든 스레드의 윈도우는 `idx = (block * block_size + thread) * V`에서 시작하고,
이는 `V = 4` 원소, 즉 16바이트의 배수입니다.

| | 128비트 트랜잭션 한 번 | 스칼라 트랜잭션 `V`번 |
|---|---|---|
| 언제 | `assumed_align=16` 약속함 | 정렬 약속 없음 |
| `a[idx:V]`가 낮아지는 것 | `ld.global.v4.b32` / `v2.b64` | `ld.global.b32` 네 개 |
| 슬라이스당 | 명령어 **1**개 | 명령어 **4**개 |

넓은 opcode는 GPU마다 다릅니다 -- Ampere/Hopper에서는 `ld.global.v4.b32`(32비트 워드 4개),
Blackwell에서는 `ld.global.v2.b64`(64비트 워드 2개)이고, 후자는 네 값을 묶어서 `add.f32x2` 하나가
두 개씩 더하게 합니다. 어느 쪽이든 같은 128비트입니다; 섹션 4가 여러분의 GPU가 쓰는 쪽을 출력합니다.

In [ ]:
@cute.kernel
def vector_add_kernel(
    a: cutlass.Array, b: cutlass.Array, c: cutlass.Array, N: cutlass.Int32, V: cutlass.Constexpr
):
    # Step 1. Map this thread to the first element of its V-wide window. The global
    # lane is (block * block_size + thread); scaling by V gives the window's start.
    tx, _, _ = cute.arch.thread_idx()
    bx, _, _ = cute.arch.block_idx()
    bdx, _, _ = cute.arch.block_dim()
    idx = (bx * bdx + tx) * V

    # Step 2. Guard the tail, then add one V-wide window. The grid rounds up to whole
    # blocks, so the last block has threads whose window starts past the array; skip them.
    if idx < N:
        c[idx:V] = a[idx:V] + b[idx:V]

## 2. Launch: 윈도우당 스레드 하나

스레드 하나가 `V`개를 맡으니 스레드가 `N / V`개 필요합니다. 1차원 블록 256스레드(관례적이고 점유율에
좋은 크기)를 고르고, 빠지는 원소가 없도록 그리드를 올림합니다 -- 남는 몇 개의 스레드는 커널 안의 범위
검사가 흡수합니다. `N`은 `V`의 배수로 잡아서, 활성 스레드의 윈도우가 전부 배열 안에 온전히 들어옵니다.

In [ ]:
@cute.jit
def vector_add(
    a: cutlass.Array, b: cutlass.Array, c: cutlass.Array, N: cutlass.Int32, V: cutlass.Constexpr
):
    block = (256, 1, 1)
    threads = N // V  # one thread per V-wide window
    grid = ((threads + block[0] - 1) // block[0], 1, 1)  # round up to whole blocks

    vector_add_kernel(a, b, c, N, V).launch(grid=grid, block=block)

## 3. 실행하고 PyTorch와 비교하기

`cutlass.Array` 매개변수는 호스트 진입 타입이라서 버퍼를 손으로 세팅할 필요가 없습니다: PyTorch CUDA
텐서를 `cute.runtime.from_dlpack`으로 커널에 그대로 넘기면, 기존 GPU 메모리를 복사 없이 감쌉니다.
그다음 결과를 PyTorch 자신의 `a + b`와 비교합니다.

In [ ]:
N, V = 1 << 20, 4  # 1,048,576 elements, 4 per thread

# The grid is sized as N // V, so V must divide N exactly. Otherwise the trailing
# elements would have no thread assigned, and a thread's window could run past the end.
assert N % V == 0, "N must be a multiple of the vector width V"

a = torch.randn(N, dtype=torch.float32, device="cuda")
b = torch.randn(N, dtype=torch.float32, device="cuda")
c = torch.zeros(N, dtype=torch.float32, device="cuda")

# from_dlpack wraps each CUDA tensor as a cutlass.Array with no copy. assumed_align=16
# promises 16-byte-aligned buffers (torch's allocator guarantees far more), which is what
# lets the V=4 slice load/store become single 128-bit transactions (see section 4).
a_ = cute.runtime.from_dlpack(a, assumed_align=16)
b_ = cute.runtime.from_dlpack(b, assumed_align=16)
c_ = cute.runtime.from_dlpack(c, assumed_align=16)

vector_add(a_, b_, c_, N, V)

torch.testing.assert_close(c.cpu(), (a + b).cpu(), atol=1e-5, rtol=1e-5)
print("PASS")

# Expected output:
# PASS

## 4. 벡터화됐음을 증명하기 -- PTX에서 직접

`V = 4` 슬라이스가 128비트 트랜잭션 한 번이 된다고 주장했습니다. `cute.compile[cute.KeepPTX]`는
생성된 PTX를 결과의 `.artifacts.PTX`에 남겨 주니, 그걸 읽어서 global load/store만 봅니다.
*같은* 커널을 정렬 약속 **있이** / **없이** 컴파일해 보면 차이가 그대로 드러납니다: 벡터 명령어
세 개 대 스칼라 명령어 열두 개. (넓은 opcode는 GPU마다 다릅니다 -- Ampere/Hopper는 `v4.b32`,
Blackwell은 `v2.b64` -- 하지만 둘 다 128비트 트랜잭션 한 번이므로 핵심은 *개수*입니다.)

In [ ]:
def global_mem_ops(compiled):
    """The global load/store instructions in a compiled kernel's PTX."""
    return [
        ln.strip()
        for ln in compiled.artifacts.PTX.splitlines()
        if "ld.global" in ln or "st.global" in ln
    ]


print("WITH the 16-byte promise (assumed_align=16):")
for op in global_mem_ops(cute.compile[cute.KeepPTX](vector_add, a_, b_, c_, N, V)):
    print("   ", op)

# Same kernel, same data -- but no alignment promise, so the compiler stays conservative.
plain = [cute.runtime.from_dlpack(t) for t in (a, b, c)]
print("\nWITHOUT the promise:")
for op in global_mem_ops(cute.compile[cute.KeepPTX](vector_add, *plain, N, V)):
    print("   ", op)

# Expected output -- the exact opcode is GPU-specific (v4.b32 on Ampere/Hopper, v2.b64 on
# Blackwell); both are one 128-bit transaction, so the *count* is the point:
# WITH the 16-byte promise (assumed_align=16):   -> 3 vector ops
#     ld.global.{v4.b32 | v2.b64} ...   <- one 128-bit load per operand (a and b)
#     ld.global.{v4.b32 | v2.b64} ...
#     st.global.{v4.b32 | v2.b64} ...   <- one 128-bit store
# WITHOUT the promise:                            -> 12 scalar ops
#     ld.global.b32 ...   <- eight 32-bit loads (four per operand)
#     st.global.b32 ...   <- four 32-bit stores

## 1. `@cute.kernel` 안의 Array

커널 안에서는 스크래치를 할당합니다: 블록마다 **shared** 타일 하나, 스레드마다 **local** 버퍼 하나.
이 순환 3-tap 행 블러가 둘 다 씁니다. **global** 인자에서 한 행을 읽어 shared에 올려두고(그래야 각
스레드가 이웃 열에 닿습니다), 탭 세 개를 local에 모은 뒤 평균을 써냅니다. 먼저 각 배열의 레이아웃을
출력합니다 — 한 번만 일어나는 컴파일 타임 출력입니다.

In [ ]:
@cute.kernel
def blur_kernel(inp: cutlass.Array, out: cutlass.Array, C: cutlass.Constexpr):
    row, _, _ = cute.arch.block_idx()   # one block per row
    col, _, _ = cute.arch.thread_idx()  # one thread per column

    # Step 1. Allocate scratch and print each array's layout (a one-time, compile-time print).
    tile = cutlass.Array(inp.dtype, C, space=cutlass.AddressSpace.smem)  # per-block scratch
    taps = cutlass.Array(inp.dtype, 3)                                   # per-thread scratch
    print("arg    :", inp)        # a global Array (the kernel argument)
    print("shared :", tile)
    print("local  :", taps)

    # Step 2. Stage this row from global into shared; the barrier makes it block-visible.
    row_in = inp.subview(row * inp.strides[0])   # subview: an offset view of this row
    tile[col] = row_in[col]                      # global -> shared
    cute.arch.barrier()                     # row now visible to every thread

    # Step 3. Gather three taps into local and write the average back.
    taps[0] = tile[(col + C - 1) % C]            # left neighbor (another thread wrote it)
    taps[1] = tile[col]
    taps[2] = tile[(col + 1) % C]                # right neighbor
    out[row, col] = (taps[0] + taps[1] + taps[2]) / 3.0

In [ ]:
@cute.jit
def blur(inp: cutlass.Array, out: cutlass.Array, N: cutlass.Int32, C: cutlass.Constexpr):
    blur_kernel(inp, out, C).launch(grid=(N, 1, 1), block=(C, 1, 1))


N, C = 1024, 256
inp = torch.randn(N, C, dtype=torch.float32, device="cuda")
out = torch.zeros_like(inp)

blur(cute.runtime.from_dlpack(inp), cute.runtime.from_dlpack(out), N, C=C)

ref = (inp.roll(1, dims=1) + inp + inp.roll(-1, dims=1)) / 3.0   # circular 3-tap blur
torch.testing.assert_close(out.cpu(), ref.cpu(), atol=1e-5, rtol=1e-5)
print("PASS")

# Expected output (the layout prints fire once, while the kernel is staged):
# arg    : Array(dtype=Float32, address_space=gmem, shape=(1024, 256), alignment=4)
# shared : Array(dtype=Float32, address_space=smem, shape=(256,), alignment=4)
# local  : Array(dtype=Float32, address_space=generic, shape=(3,), alignment=4)
# PASS
#
# Only the local array reports `generic`, and that is correct -- not a mislabel: per-thread
# scratch is an `alloca`, stack memory addressed through a *generic* pointer, so one load/store
# path serves it. The dedicated spaces (gmem/smem/cmem) each carry their own address space and
# report it by name.

## 2. `@cute.kernel` 밖의 Array

**global**과 **constant** 배열은 프로그램 전체의 정적 변수이므로, CUDA C++의 `__device__` /
`__constant__`처럼 **모듈 스코프** — 어떤 커널 밖 — 에서 한 번 선언하고 이름으로 씁니다.
(모듈 스코프 정적 변수는 `import cutlass`만 하면 바로 동작합니다 — 별도 설정이 필요 없습니다.)

`GAMMA`는 읽기 전용 상수 조회표(`init=`으로 값을 굽습니다)이고, `RUN_COUNT`는 **launch를 가로질러
유지되는** global 카운터입니다. 커널은 입력에 `GAMMA`를 곱하고 `RUN_COUNT`를 올립니다. 정적 변수는
인덱싱하면 `Array`가 되는 `_GlobalVariable` 핸들이라서, `.subview(0)`(버퍼 전체)을 출력해 `Array`로
보이게 합니다.

In [ ]:
# Declared OUTSIDE any kernel, at module scope:
GAMMA = cutlass.Array(cutlass.Float32, 4, name="gamma",
                      init=[1.0, 1.5, 2.0, 2.5], space=cutlass.AddressSpace.cmem)
RUN_COUNT = cutlass.Array(cutlass.Int32, 1, name="run_count", space=cutlass.AddressSpace.gmem)


@cute.kernel
def scale_kernel(inp: cutlass.Array, out: cutlass.Array, seen: cutlass.Array):
    i, _, _ = cute.arch.thread_idx()

    # Step 1. Materialize each module-scope static as an Array and print its layout.
    print("constant:", GAMMA.subview(0))     # materialize the static -> Array
    print("global  :", RUN_COUNT.subview(0))

    # Step 2. Scale the input by the module-scope constant table.
    out[i] = inp[i] * GAMMA[i]                          # read the module-scope constant

    # Step 3. Thread 0 bumps the module-scope global counter (it persists across launches).
    if i == 0:
        RUN_COUNT[0] = RUN_COUNT[0] + cutlass.Int32(1)  # bump the module-scope global counter
        seen[0] = RUN_COUNT[0]


@cute.jit
def scale(inp: cutlass.Array, out: cutlass.Array, seen: cutlass.Array):
    scale_kernel(inp, out, seen).launch(grid=(1, 1, 1), block=(4, 1, 1))

In [ ]:
g_in = torch.tensor([2.0, 4.0, 6.0, 8.0], device="cuda")
g_out = torch.zeros(4, device="cuda")
seen = torch.zeros(1, dtype=torch.int32, device="cuda")

scale(cute.runtime.from_dlpack(g_in), cute.runtime.from_dlpack(g_out), cute.runtime.from_dlpack(seen))

torch.testing.assert_close(g_out.cpu(), torch.tensor([2.0, 6.0, 12.0, 20.0]))  # inp * GAMMA
assert seen.item() == 1                                                         # counter went 0 -> 1
print("PASS")

# Expected output (layout prints fire once, while the kernel is staged):
# constant: Array(dtype=Float32, address_space=cmem, shape=(4,), alignment=4)
# global  : Array(dtype=Int32, address_space=gmem, shape=(1,), alignment=4)
# PASS

## 5. 보너스 — 같은 커널을 데코레이터로

위의 커널 + launch는 *어떤* 원소별 연산이든 똑같은 상용구입니다. 작은 `@elementwise` **데코레이터**가
그걸 한 번에 담아서, 한 줄짜리 연산이 곧바로 완전한 벡터화 커널이 됩니다. 연산은 트레이스 시점에 도는
평범한 파이썬입니다 -- 여기서는 `x + y`가 똑같은 `V`-폭 덧셈을, 슬라이스 load/store까지 그대로 만들어
냅니다.

In [ ]:
def elementwise(op):
    """Lift a binary op into a vectorized kernel + launcher (one V-wide window per thread)."""

    @cute.kernel
    def kern(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array,
             N: cutlass.Int32, V: cutlass.Constexpr):
        tx, _, _ = cute.arch.thread_idx()
        bx, _, _ = cute.arch.block_idx()
        bdx, _, _ = cute.arch.block_dim()
        idx = (bx * bdx + tx) * V
        if idx < N:
            c[idx:V] = op(a[idx:V], b[idx:V])   # op runs at trace time, on the V-wide vectors

    @cute.jit
    def host(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array,
             N: cutlass.Int32, V: cutlass.Constexpr):
        threads = N // V
        grid = ((threads + 255) // 256, 1, 1)
        kern(a, b, c, N, V).launch(grid=grid, block=(256, 1, 1))

    return host


@elementwise
def vadd(x, y):
    return x + y   # the whole op -- the decorator supplies the kernel and the launch


vadd(a_, b_, c_, N, V)
torch.testing.assert_close(c.cpu(), (a + b).cpu(), atol=1e-5, rtol=1e-5)
print("PASS  (decorator-generated kernel matches)")

# Expected output:
# PASS  (decorator-generated kernel matches)

### 실측 검증 (2026-10-01, Colab L4)

이 노트북 전체를 `.ipynb` 단위로(실제 셀 순서대로 하나의 커널에서) Colab **L4**에서 실행해
확인했습니다. 코드 셀 13개가 전부 에러 없이 통과했습니다(`nvidia-cutlass-dsl==4.8.0`). 노트북4처럼
`CUTE_DSL_ARCH` 오버라이드 없이 기본 감지(sm_89)로 돌렸고, **코드 셀은 전부 원문 그대로**입니다
(`cute.printf`를 쓰지 않아 `show_device_printf()`도 필요 없었습니다).

| 절 | 확인한 출력 | 원문 기대 출력과 |
|---|---|---|
| 3 벡터 덧셈 | `PASS` | 일치 |
| 4 PTX 증명 | 정렬 약속 **있이** `ld.global.v4.b32` ×2 + `st.global.v4.b32` ×1 = **3개**, **없이** `ld.global.b32` ×8 + `st.global.b32` ×4 = **12개** | 일치 (L4에서도 `v4.b32`) |
| 블러(커널 안 Array) | `arg: ...address_space=gmem, shape=(1024, 256), alignment=4`, `shared: ...smem, shape=(256,), alignment=4`, `PASS` | 앞 두 줄·`PASS` 일치 |
| **블러 — local 배열** | `local : Array(dtype=Float32, address_space=`**`rmem`**`, shape=(3,), alignment=`**`1024`**`)` | **불일치** (아래) |
| 정적 변수(커널 밖 Array) | `constant: ...cmem, shape=(4,), alignment=4`, `global  : ...gmem (Int32), shape=(1,)`, `PASS` (카운터 0→1 assert 통과) | 일치 |
| 보너스 데코레이터 | `PASS  (decorator-generated kernel matches)` | 일치 |

**원문과 다른 점 — local 배열의 공간 표기:** 블러 커널 셀 끝의 원문 주석은 local 배열이
`address_space=generic, alignment=4`로 출력되고 "`generic`이 맞다(mislabel 아님)"고 설명합니다. 이 L4 실행
(휠 `4.8.0`)에서는 `address_space=rmem, alignment=1024`가 나왔습니다. 고정 커밋(`098de2a` = v4.8.0 태그)의
`base_dsl/array.py`에는 `rmem` 텐서를 만드는 경로(`:1167`~`:1186`)와 generic 포인터로 `alloca`하는 경로(`:1196`,
`:1206`)가 둘 다 있습니다. 기본 공간에서 어느 경로가 실행되는지, 왜 값이 다른지는 확인하지 않았습니다. 코드 셀은
원문 그대로 두었기 때문에 그 주석은 이 노트북에도 남아 있지만, **이 실행에서 실제로 출력된 값은 위의
`rmem` / `1024`입니다.**

**표기 참고:** 원문은 섹션 번호가 "네 가지 공간 → 1~4 → 다시 1, 2 → 5(보너스)"로 중복됩니다. 원문 그대로 유지했습니다.

**검증의 한계:** L4(sm_89)에서 한 번 실행한 결과입니다. 다른 GPU에서의 opcode(`v2.b64` 등)는 확인하지 않았고,
"직접 해보기"의 변형들은 실행하지 않았습니다.


## 직접 해보기

1. **배리어 빼기.** `blur_kernel`에서 `cute.arch.barrier()`를 지워 보세요 — 이웃 스레드가 쓰기 전에
   그 열을 읽을 수 있어서, 블러가 틀리고 비결정적이 됩니다.
2. **타일을 local로 만들기.** `tile`에서 `space=…smem`을 빼고(배리어도 함께) 보세요: 스레드마다
   *자기만의* `tile`을 갖고 거기엔 `tile[col]`만 써져 있으니, 이웃 읽기는 초기화되지 않은 메모리를
   건드립니다 — shared(블록 전체)와 local(스레드별)의 차이입니다.
3. **안쪽이냐 바깥쪽이냐.** `GAMMA`의 선언을 `scale_kernel` *안으로* 옮겨 보세요 — 그래도 동작합니다:
   이름 붙은 정적 변수는 어디서 선언하든 같은 버퍼입니다. 그다음 `scale`을 두 번 launch하고
   `RUN_COUNT`가 2까지 올라가는 걸 보세요 — global 정적 변수는 launch를 가로질러 유지됩니다.
4. **constant는 읽기 전용.** 커널에서 `GAMMA`에 저장해 보세요 — `TypeError`가 납니다, constant
   메모리는 디바이스에서 쓸 수 없으니까요. (그리고 `AddressSpace` enum의 `tmem` / `dsmem`은
   `cutlass.Array`로 쓸 수 없습니다; Blackwell 챕터에 전용 API가 있습니다.)